In [0]:
from pyspark.sql import functions as F

o = spark.table("olist.silver.orders")
i = spark.table("olist.silver.order_items")
c = spark.table("olist.silver.customers")
r = spark.table("olist.silver.reviews")

order_totals = (i.groupBy("order_id").agg(
    F.sum("price").alias("revenue"),
    F.sum("freight").alias("freight_cost"),
    F.count("*").alias("item_count"),
    F.countDistinct("seller_id").alias("seller_count"),
    F.first("seller_id").alias("primary_seller_id")))

fact = (o.filter("is_delivered AND delivered_at IS NOT NULL")
    .join(order_totals, "order_id", "left")
    .join(c, "customer_id", "left")
    .join(r.select("order_id", F.col("score").alias("review_score")), "order_id", "left")
    .select(
        "order_id", "customer_key", "primary_seller_id",
        F.col("state").alias("customer_state"), "city",
        "purchased_at", "delivered_at", "promised_at",
        "delivery_days", "days_late", "is_late", "approval_hours",
        "revenue", "freight_cost", "item_count", "seller_count", "review_score")
    .withColumn("order_month", F.date_trunc("month", "purchased_at")))

(fact.write.mode("overwrite").option("overwriteSchema", "true")
     .partitionBy("order_month")
     .saveAsTable("olist.gold.fact_order_delivery"))

In [0]:
seller_perf = (fact
    .join(spark.table("olist.silver.sellers")
            .select(F.col("seller_id").alias("primary_seller_id"),
                    F.col("state").alias("seller_state")),
          "primary_seller_id", "left")
    .groupBy("primary_seller_id", "seller_state")
    .agg(
        F.count("*").alias("orders"),
        F.round(F.avg(F.col("is_late").cast("int")) * 100, 1).alias("late_pct"),
        F.round(F.avg("delivery_days"), 1).alias("avg_delivery_days"),
        F.round(F.avg("review_score"), 2).alias("avg_review"),
        F.round(F.sum("revenue"), 2).alias("revenue"))
    .filter("orders >= 30")            # ignore noise from tiny sellers
    .withColumn("on_time_pct", 100 - F.col("late_pct")))

seller_perf.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("olist.gold.agg_seller_performance")

In [0]:
state_perf = (fact.groupBy("customer_state").agg(
    F.count("*").alias("orders"),
    F.round(F.avg("delivery_days"), 1).alias("avg_delivery_days"),
    F.round(F.avg(F.col("is_late").cast("int")) * 100, 1).alias("late_pct"),
    F.round(F.avg("review_score"), 2).alias("avg_review"),
    F.round(F.avg("freight_cost"), 2).alias("avg_freight")))

state_perf.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("olist.gold.agg_state_delivery")

# monthly trend for the headline chart
monthly = (fact.groupBy("order_month").agg(
    F.count("*").alias("orders"),
    F.round(F.avg(F.col("is_late").cast("int")) * 100, 1).alias("late_pct"),
    F.round(F.avg("delivery_days"), 1).alias("avg_delivery_days"),
    F.round(F.avg("review_score"), 2).alias("avg_review"))
    .orderBy("order_month"))

monthly.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("olist.gold.agg_monthly_trend")

In [0]:
%sql
SHOW TABLES IN olist.gold
